## ___Zoom meeting 10-09-2026 with Aki___
---------------------

In [1]:
suppressPackageStartupMessages({
    library("loo")
    library("cmdstanr")
    library("posterior")
    library("ggplot2")
})

source("../Aki/R/loo_compare_scales.R") # code by Aki

### ___Model specifications___
-------

```R
M1 <- (F00727, F00679, F00709) ~ state + (1|gr(binominal, cov = corrmat)) + (1|taxa)    
M2 <- (F00727, F00679, F00709) ~ state + (1|taxa))    
M3 <- (F00727, F00679, F00709) ~ state + (1|p|gr(binominal, cov = corrmat)) + (1|q|taxa)    
M4 <- (F00727, F00679, F00709) ~ state + (1|q|taxa))    
 
N1 <- (F00727, F00679, F00709) ~ 1 + (1|gr(binominal, cov = corrmat)) + (1|taxa)     
N2 <- (F00727, F00679, F00709) ~ 1 + (1|taxa))       
N3 <- (F00727, F00679, F00709) ~ 1 + (1|p|gr(binominal, cov = corrmat)) + (1|q|taxa) 
N4 <- (F00727, F00679, F00709) ~ 1 + (1|q|taxa))            
```

Paired comparison (when looking at the effect of going from M to N, to get proper SE diff, the comparisons need to be paired).

| pair | phylogeny? | elpd diff | SE diff |
|----------|------------|----------:|--------:|
| M1 vs N1 | yes | 4.8 | 0.7 |
| M3 vs N3 | yes | 6.1 | 1.1 |
| M2 vs N2 | no | 41.6 | 2.7 |
| M4 vs N4 | no | 34.4 | 3.4 |

Once the phylogeny accounts for the trait covariance, `state` has
nothing left to explain. 

In [ ]:
# is this because of the small expected log predictive density (ELPD) differences between the M and N models??

# https://users.aalto.fi/%7Eave/CV-FAQ.html#elpd_interpretation
# https://users.aalto.fi/%7Eave/CV-FAQ.html#se_diff

`state` is species-constant and its apparent
effect in M2/M4 is confounded with phylogeny.

For scale, the structural terms are worth an order of magnitude more
than `state`: phylogeny is 481 nats (M3 vs M4) and cross-response
correlation 560 nats (M3 vs M1).

In [ ]:
# how do we measure that?? what is nats??

In [ ]:
# heaps of the following warning during model fits???

# Chain 1 Informational Message: The current Metropolis proposal is about to be rejected because of the following issue:
# Chain 1 Exception: lkj_corr_cholesky_lpdf: Random variable[2] is 0, but must be positive! (in '/tmp/Rtmp097Ci9/model-482935dea2221.stan', line 146, column 2 to column 48)
# Chain 1 If this warning occurs sporadically, such as for highly constrained variable types like covariance matrices, then the sampler is fine,
# Chain 1 but if this warning occurs often then your model may be either severely ill-conditioned or misspecified.

In [ ]:
# quite common - okay in the beginning (at the beginning underflows or overflows)
# as long as this only happens in the beginning its okay
# initializing the Markov chains - it's arbitrary initialization (unconstrained space) - transformed into a constrained space

In [2]:
MODEL_PAIRS <- list(c("M1", "N1"), c("M2", "N2"), c("M3", "N3"), c("M4", "N4"))

## The original responses, needed for the cross-scale comparison.
resps <- as.matrix(read.csv("../Aki/data/stan.csv", stringsAsFactors = FALSE)[, c("F00727", "F00679", "F00709")])

results <- list()

i <- 1
# compare every M model with its corresponding N model
for (pair in MODEL_PAIRS){
    # code for pairwise comparison
    mods <- list(readRDS(sprintf("../Aki/output/%s_loo.Rds", pair[1])), # M model
                    readRDS(sprintf("../Aki/output/%s_loo.Rds", pair[2]))) # corresponding N model
    names(mods) <- pair # name the list elements properly
    results[[i]] <- loo_compare_original_scale(mods, resps, TRUE)
    i <- i + 1
}

In [18]:
rbind(results[[1]], results[[2]], results[[3]], results[[4]])

model,elpd_diff,se_diff,p_worse,diag_diff,diag_elpd,elpd_loo,se_elpd_loo,p_loo,se_p_loo,looic,se_looic
<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
M1,0.000000,0.000000,NA,,1 k_psis > 0.7,-12319.36,280.8319,52.40050,7.667561,14587.10,570.6892
N1,-4.116264,6.580517,0.7341860,,,-12323.48,280.5885,45.01772,7.300608,14595.33,570.0834
M2,0.000000,0.000000,NA,,,-13129.52,273.5878,56.55952,7.451734,16207.42,553.8773
N2,-42.208853,15.540058,0.9966975,,2 k_psis > 0.7,-13171.73,273.8020,45.38473,7.409197,16291.84,554.0556
M3,0.000000,0.000000,NA,,1 k_psis > 0.7,-11758.78,298.2867,62.42159,7.953015,13465.94,608.2570
N3,-4.598388,5.530124,0.7971590,,,-11763.38,298.4793,55.29608,7.587726,13475.13,608.5821
M4,0.000000,0.000000,NA,,,-12239.33,299.3387,61.58090,7.681779,14427.03,610.1229
N4,-34.373169,10.211147,0.9996190,,,-12273.70,299.9433,51.78996,7.752676,14495.77,611.1913


| pair | phylogeny | ELPD diff | SE diff |
|----------|------------|----------:|--------:|
| M1 vs N1 | TRUE | 6.580517 | 6.580517	 |
| M3 vs N3 | TRUE | -4.598388 | 5.530124 |
| M2 vs N2 | FALSE | -42.20885 | 15.54006 |
| M4 vs N4 | FALSE | -34.37317 | 10.21115 |


In [4]:
# see what happens when the models account for possible cross trait correlations

MODEL_PAIRS <- list(c("M1", "M3"), c("M2", "M4"), c("N1", "N3"), c("N2", "N4"))
resps <- as.matrix(read.csv("../Aki/data/stan.csv", stringsAsFactors = FALSE)[, c("F00727", "F00679", "F00709")])
results <- list()
i <- 1

for (pair in MODEL_PAIRS){
    mods <- list(readRDS(sprintf("../Aki/output/%s_loo.Rds", pair[1])), # M model
                    readRDS(sprintf("../Aki/output/%s_loo.Rds", pair[2]))) # corresponding N model
    names(mods) <- pair
    results[[i]] <- loo_compare_original_scale(mods, resps, TRUE)
    i <- i + 1
}

In [5]:
rbind(results[[1]], results[[2]], results[[3]], results[[4]])

model,elpd_diff,se_diff,p_worse,diag_diff,diag_elpd,elpd_loo,se_elpd_loo,p_loo,se_p_loo,looic,se_looic
<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
M3,0.0000,0.00000,NA,,1 k_psis > 0.7,-11758.78,298.2867,62.42159,7.953015,13465.94,608.2570
M1,-560.5816,59.60634,1,,1 k_psis > 0.7,-12319.36,280.8319,52.40050,7.667561,14587.10,570.6892
M4,0.0000,0.00000,NA,,,-12239.33,299.3387,61.58090,7.681779,14427.03,610.1229
M2,-890.1966,76.87164,1,,,-13129.52,273.5878,56.55952,7.451734,16207.42,553.8773
N3,0.0000,0.00000,NA,,,-11763.38,298.4793,55.29608,7.587726,13475.13,608.5821
N1,-560.0995,60.38188,1,,,-12323.48,280.5885,45.01772,7.300608,14595.33,570.0834
N4,0.0000,0.00000,NA,,,-12273.70,299.9433,51.78996,7.752676,14495.77,611.1913
N2,-898.0323,78.24755,1,,2 k_psis > 0.7,-13171.73,273.8020,45.38473,7.409197,16291.84,554.0556


In [6]:
# see what happens when the model incorporates a phylogenetic random effect

MODEL_PAIRS <- list(c("M1", "M2"), c("M3", "M4"), c("N1", "N2"), c("N3", "N4"))
resps <- as.matrix(read.csv("../Aki/data/stan.csv", stringsAsFactors = FALSE)[, c("F00727", "F00679", "F00709")])
results <- list()
i <- 1

for (pair in MODEL_PAIRS){
    mods <- list(readRDS(sprintf("../Aki/output/%s_loo.Rds", pair[1])), # M model
                    readRDS(sprintf("../Aki/output/%s_loo.Rds", pair[2]))) # corresponding N model
    names(mods) <- pair
    results[[i]] <- loo_compare_original_scale(mods, resps, TRUE)
    i <- i + 1
}

In [7]:
rbind(results[[1]], results[[2]], results[[3]], results[[4]])

model,elpd_diff,se_diff,p_worse,diag_diff,diag_elpd,elpd_loo,se_elpd_loo,p_loo,se_p_loo,looic,se_looic
<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
M1,0.0000,0.00000,NA,,1 k_psis > 0.7,-12319.36,280.8319,52.40050,7.667561,14587.10,570.6892
M2,-810.1613,58.38694,1,,,-13129.52,273.5878,56.55952,7.451734,16207.42,553.8773
M3,0.0000,0.00000,NA,,1 k_psis > 0.7,-11758.78,298.2867,62.42159,7.953015,13465.94,608.2570
M4,-480.5463,41.73836,1,,,-12239.33,299.3387,61.58090,7.681779,14427.03,610.1229
N1,0.0000,0.00000,NA,,,-12323.48,280.5885,45.01772,7.300608,14595.33,570.0834
N2,-848.2539,59.79359,1,,2 k_psis > 0.7,-13171.73,273.8020,45.38473,7.409197,16291.84,554.0556
N3,0.0000,0.00000,NA,,,-11763.38,298.4793,55.29608,7.587726,13475.13,608.5821
N4,-510.3211,42.54921,1,,,-12273.70,299.9433,51.78996,7.752676,14495.77,611.1913


In [ ]:
# look up the below link to see how to interpret loo::loo() results
# https://mc-stan.org/loo/reference/loo_compare.html

In [17]:
M1 <- readRDS("../Aki/output/M1_fit.Rds")
M2 <- readRDS("../Aki/output/M2_fit.Rds")
M3 <- readRDS("../Aki/output/M3_fit.Rds")

In [8]:
posterior::as_draws_rvars(M1)

# A draws_rvars: 1000 iterations, 8 chains, and 17 variables
$lp__: rvar<1000,8>[1] mean ± sd:
[1] -8233 ± 3.9 

$b: rvar<1000,8>[5,3] mean ± sd:
     [,1]             [,2]             [,3]            
[1,] -0.1275 ± 0.174   0.0075 ± 0.082   0.2902 ± 0.130 
[2,]  0.8616 ± 0.221  -0.2802 ± 0.103  -0.1979 ± 0.160 
[3,]  1.1681 ± 0.514  -0.9354 ± 0.237   0.4657 ± 0.375 
[4,]  1.5905 ± 0.352  -0.4804 ± 0.163  -0.0144 ± 0.238 
[5,]  0.4897 ± 0.150  -0.1806 ± 0.072  -0.0814 ± 0.107 

$Intercept: rvar<1000,8>[3] mean ± sd:
[1]  3.39 ± 0.51  -0.77 ± 0.23  -1.68 ± 0.37 

$sigma: rvar<1000,8>[3] mean ± sd:
[1] 0.74 ± 0.0085  0.34 ± 0.0039  0.40 ± 0.0044 

$Lrescor: rvar<1000,8>[3,3] mean ± sd:
     [,1]            [,2]            [,3]           
[1,]  1.00 ± 0.0000   0.00 ± 0.0000   0.00 ± 0.0000 
[2,] -0.75 ± 0.0070   0.67 ± 0.0078   0.00 ± 0.0000 
[3,] -0.52 ± 0.0113  -0.49 ± 0.0105   0.70 ± 0.0077 

$sd_t: rvar<1000,8>[3] mean ± sd:
[1] 0.040 ± 0.0272  0.089 ± 0.0085  0.190 ± 0.0127 

$sd_p: 

In [4]:
# we get three values because these models are TRIVARIATE
# in Aki's implementation the order of traits is => c("F00727", "F00679", "F00709")

posterior::as_draws_rvars(M1)$sd_t # non-phylogenetic random effect (taxa)

rvar<1000,8>[3] mean ± sd:
[1] 0.040 ± 0.0272  0.089 ± 0.0085  0.190 ± 0.0127 

In [6]:
posterior::as_draws_rvars(M3)$sd_t

rvar<1000,8>[3] mean ± sd:
[1] 0.43 ± 0.032  0.21 ± 0.013  0.28 ± 0.016 

In [5]:
posterior::as_draws_rvars(M1)$sd_p # phylogenetic random effect (binominal)

rvar<1000,8>[3] mean ± sd:
[1] 1.22 ± 0.053  0.53 ± 0.029  0.89 ± 0.046 

In [7]:
posterior::as_draws_rvars(M3)$sd_p

rvar<1000,8>[3] mean ± sd:
[1] 1.50 ± 0.103  0.62 ± 0.047  0.91 ± 0.059 

In [16]:
posterior::as_draws_rvars(M1)$sd_t

rvar<1000,8>[3] mean ± sd:
[1] 0.040 ± 0.0272  0.089 ± 0.0085  0.190 ± 0.0127 

In [18]:
posterior::as_draws_rvars(M2)$sd_t # no phylogenetic random effect

rvar<1000,8>[3] mean ± sd:
[1] 0.59 ± 0.021  0.26 ± 0.009  0.47 ± 0.013 

In [8]:
posterior::as_draws_rvars(M1)$Intercept # don't really need this here

rvar<1000,8>[3] mean ± sd:
[1]  3.39 ± 0.51  -0.77 ± 0.23  -1.68 ± 0.37 

In [15]:
posterior::as_draws_rvars(M1) |> attributes() # don't see L_p and L_t here?????

$names
 [1] "lp__"        "b"           "Intercept"   "sigma"       "Lrescor"    
 [6] "sd_t"        "sd_p"        "LSigma_r"    "LSigma_t"    "LSigma_p"   
[11] "lprior"      "b_Intercept" "Rescor"      "Sigma_t_out" "Sigma_p_out"
[16] "lp_between"  "log_lik"    

$class
[1] "draws_rvars" "draws"       "list"